# Longformer with a classification head

Fine-tunes `allenai/longformer-base-4096` to decide whether two texts were written by the same
author. The hyperparameters are tuned with Optuna on a sample of the training pairs. The final
model is trained on the training pairs, its threshold is selected on the validation pairs and
it is evaluated once on the test pairs.

A restarted run continues where it stopped: the study lives in a SQLite file, every training
writes checkpoints and a finished run is not repeated. A study is never continued with other
settings than the ones it was started with.

In [ ]:
SMOKE = False  # True - a few rows of every data set, a checkpoint after every step
N_TRIALS = 30  # finished trials of the study (complete; nothing is pruned)
SEED = 42  # seed of the tuning sample, the sampler, the new head and the training order
RUN_TAG = ""  # label in the file names of a repeated run, so two runs never share a file
MODEL_ID = "allenai/longformer-base-4096"  # Hugging Face id, or the folder of a saved model
DEVICE = "cuda"  # "cpu" only for the CPU checks - there is no fallback to the CPU

Standard libraries

In [ ]:
import gc
import logging
import re
from functools import partial
from pathlib import Path

Data, plots and Optuna

In [ ]:
import matplotlib.pyplot as plt
import optuna
import pandas as pd
import seaborn as sns
from optuna.trial import TrialState
from sklearn.metrics import confusion_matrix

Torch and Hugging Face

In [ ]:
import torch
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    set_seed,
)

The shared modules of the thesis

In [ ]:
from avcore import data, evaluation, runs, tracking, transformer_training, tuning

# Configuration

The model - its name, the extra load options and the classification type

In [ ]:
MODEL_NAME = "longformer-base-4096"
# extra from_pretrained options of the model, also part of the study settings
MODEL_LOAD_KWARGS = {}
CLASSIFICATION_TYPE = "classification_head"
DEVELOPER_INITIALS = "JP"

What the run is, in the words of the result tables

In [ ]:
RUN_IDENTITY = {
    "family": "transformer",
    "model": MODEL_NAME,
    "representation": "none",
    "combination": "none",
    "classifier": CLASSIFICATION_TYPE,
}

Search space - the tuned hyperparameters, the fixed warmup, the pruner and the threshold grid

In [ ]:
LEARNING_RATE_RANGE = (1e-6, 5e-5)
EPOCHS_RANGE = (1, 3)
BATCH_SIZE_CHOICES = [2, 4, 8]
WEIGHT_DECAY_RANGE = (0.0, 0.1)
# not searched - the warmup of the classification fine-tuning of the model papers
WARMUP_RATIO = 0.1
# no pruning - epochs is searched, so a pruner could only decide after epoch 2
PRUNER = optuna.pruners.NopPruner()
CLASSIFICATION_THRESHOLDS = [x / 1000 for x in range(100, 999)]

Sizes - the tuning sample, the checkpoint interval and the smoke run

In [ ]:
TRAIN_TUNING_SIZE = 19_999
CHECKPOINT_MINUTES = transformer_training.CHECKPOINT_MINUTES
FINAL_CHECKPOINT_KEY = "final"
SMOKE_ROWS = {"train": 32, "validation": 8, "test": 12}
if SMOKE:
    TRAIN_TUNING_SIZE = 16
    CHECKPOINT_MINUTES = 0

The data files

In [ ]:
# the notebook runs from its own folder, two levels below the experiments folder
PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data"
TRAIN_PATH = DATA_DIR / "processed" / data.TRAIN_CLEANED_FILE
VALIDATION_PATH = DATA_DIR / "processed" / data.VALIDATION_CLEANED_FILE
TEST_PATH = DATA_DIR / "processed" / data.TEST_CLEANED_FILE
DATA_PATHS = [TRAIN_PATH, VALIDATION_PATH, TEST_PATH]

The run name - it is part of every file name of the run

In [ ]:
run_name_parts = [DEVELOPER_INITIALS, MODEL_NAME, CLASSIFICATION_TYPE]
if SMOKE:
    run_name_parts.append("smoke")
if RUN_TAG:
    run_name_parts.append(str(RUN_TAG))
RUN_NAME = "_".join([*run_name_parts, "v2"])
if not re.fullmatch(r"[\w.-]+", RUN_NAME):
    raise ValueError(
        f"Run name {RUN_NAME!r} is not a plain file name - change RUN_TAG."
    )

Every file the run writes


In [ ]:
STUDY_PATH = DATA_DIR / "studies" / f"optuna_study_{RUN_NAME}.db"
MODEL_DIR = DATA_DIR / "models" / f"{RUN_NAME}_best_model"
# checkpoints and results are written outside the repository; the result tables are built from
# results_v2, so a smoke run writes into a folder of its own, one per RUN_TAG
CHECKPOINT_DIR = Path.home() / "checkpoints_v2" / RUN_NAME
RESULTS_DIR = Path.home() / "results_v2"
if SMOKE:
    RESULTS_DIR = Path.home() / "results_v2_smoke" / str(RUN_TAG)
RUN_DIR = RESULTS_DIR / RUN_NAME
EXPERIMENT_NAME = tracking.SMOKE_EXPERIMENT_NAME if SMOKE else tracking.EXPERIMENT_NAME

Run context - the commit, the hashes of the lock file and of the data, the versions and the GPU.
It refuses a repository with uncommitted or unpushed changes.

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s")
# the training reports every epoch and every resume at this level
logging.getLogger("avcore").setLevel(logging.INFO)

device = torch.device(DEVICE)
# an unusable device stops the run here, before anything is read or written
torch.empty(0, device=device)
run_context = tracking.collect_run_context(PROJECT_DIR, DATA_PATHS)
print(f"Run {RUN_NAME} on {device}, commit {run_context.environment['commit'][:7]}")

# Load data

In [ ]:
train_df = data.read_jsonl(TRAIN_PATH)
validation_df = data.read_jsonl(VALIDATION_PATH)
test_df = data.read_jsonl(TEST_PATH)
# a prediction table is keyed by the pair id - checked on the whole files, before any training
for name, pairs_df in {"validation": validation_df, "test": test_df}.items():
    if not pairs_df["id"].is_unique:
        raise ValueError(f"The {name} pairs have repeated ids.")
if SMOKE:
    train_df = train_df.head(SMOKE_ROWS["train"])
    validation_df = validation_df.head(SMOKE_ROWS["validation"])
    test_df = test_df.head(SMOKE_ROWS["test"])

#### Create a subset of training data for finetuning

In [ ]:
tuning_df = train_df.sample(n=TRAIN_TUNING_SIZE, random_state=SEED).reset_index(
    drop=True
)
print(
    f"train {len(train_df)}, tuning {len(tuning_df)}, validation {len(validation_df)}, test {len(test_df)}"
)

# Set up model

The tokenizer and the config of the pretrained model are loaded once.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
pretrained_config = AutoConfig.from_pretrained(MODEL_ID)

The model is loaded again for every trial and for the final training, with the seed set first, so its new classification head always starts from the same weights. Every dropout stays as in the config of the pretrained model.

In [ ]:
def build_model(device: torch.device) -> PreTrainedModel:
    """Load the pretrained model with a new 2-label classification head.

    Args:
        device: Device the model is moved to.

    Returns:
        The model on the device, 0 = different author, 1 = same author.
    """
    # the new head gets the same weights on every call
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=2,
        problem_type="single_label_classification",
        **MODEL_LOAD_KWARGS,
    )
    return model.to(device)

# Hyperparameter optimization

Objective function - trains one trial on the tuning sample, reports the weighted F1 after
every epoch (no pruning) and returns the unrounded F1 at the best classification threshold on the
validation pairs.

In [ ]:
def objective(
    trial: optuna.Trial,
    *,
    tokenizer: PreTrainedTokenizerBase,
    tuning_df: pd.DataFrame,
    validation_df: pd.DataFrame,
    device: torch.device,
) -> float:
    """Train and evaluate one trial.

    Args:
        trial: Trial started by tuning.run_study.
        tokenizer: Tokenizer of the model.
        tuning_df: Training pairs of the tuning.
        validation_df: Pairs for the per-epoch report and the threshold search.
        device: Device of the training.

    Returns:
        The unrounded F1 (a fraction) at the best classification threshold on the validation
        pairs; the rounded percent metrics are stored as user attributes.
    """
    learning_rate = trial.suggest_float("lr", *LEARNING_RATE_RANGE, log=True)
    epochs = trial.suggest_int("epochs", *EPOCHS_RANGE)
    batch_size = trial.suggest_categorical("batch_size", BATCH_SIZE_CHOICES)
    weight_decay = trial.suggest_float("weight_decay", *WEIGHT_DECAY_RANGE)
    settings = transformer_training.TrainingSettings(
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        epochs=epochs,
        batch_size=batch_size,
        warmup_ratio=WARMUP_RATIO,
        seed=SEED,
    )

    # a trial that ended with an exception leaves its model in a reference cycle of it
    gc.collect()
    torch.cuda.empty_cache()
    model = build_model(device)
    result = transformer_training.train_classifier(
        model=model,
        tokenizer=tokenizer,
        train_df=tuning_df,
        settings=settings,
        checkpoint_dir=CHECKPOINT_DIR,
        checkpoint_key=transformer_training.checkpoint_key_for(trial),
        validation_df=validation_df,
        trial=trial,
        checkpoint_minutes=CHECKPOINT_MINUTES,
    )
    del model
    gc.collect()
    torch.cuda.empty_cache()

    validation_scores_df = result.validation_scores_df
    threshold_results_df = evaluation.evaluate_classification_thresholds(
        validation_scores_df["same"].astype(int),
        validation_scores_df["score"],
        CLASSIFICATION_THRESHOLDS,
    )
    best_row = evaluation.select_best_threshold(threshold_results_df)
    trial.set_user_attr("threshold", float(best_row["threshold"]))
    for metric in evaluation.METRIC_COLUMNS:
        trial.set_user_attr(f"validation_{metric}", float(best_row[metric]))
    print(
        f"Trial {trial.number}: validation F1 {best_row['f1']} at threshold {best_row['threshold']}"
    )
    return float(best_row["f1_unrounded"])